# Chạy phần mạng nơ-ron trên Colab

Notebook này chỉ chạy **RNN/LSTM**: tune, rồi CV bản chính (5 seed), rồi CV bản `_ar` (3 seed). Các model thống kê và cây chạy trên laptop, vì laptop nhanh hơn. Xong thì gộp kết quả về laptop bằng `scripts/merge_colab.py`.

**Trước khi chạy:**
1. Trên laptop, `git push origin main` để Colab lấy được code mới nhất.
2. Trong Google Drive tạo thư mục `MyDrive/rsf/data/`, rồi upload 2 file từ `D:\retail-sales-forecasting-research\data\processed\`: `daily_series.csv` và `promotions.csv`.
3. Runtime → Change runtime type → **CPU**. GPU không giúp gì, vì code chạy torch trên CPU.

**Khi đang chạy:**
- Giữ tab mở. Colab bản miễn phí ngắt khi tab bỏ không lâu, và mỗi phiên tối đa khoảng 12 giờ.
- Bị ngắt thì chạy lại từ cell đầu. Mỗi task ghi một file vào Drive, nên task đã xong sẽ được bỏ qua.
- Không chạy stage `test` ở đây.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/rsf'
os.makedirs(f'{BASE}/results', exist_ok=True)
os.environ['RSF_PROCESSED_DIR'] = f'{BASE}/data'
os.environ['RSF_RESULTS_DIR'] = f'{BASE}/results'
os.environ['RSF_TORCH_THREADS'] = '1'   # Colab có 2 lõi, chạy --jobs 2 nên mỗi tiến trình 1 luồng
os.environ['PYTHONUTF8'] = '1'
print(os.listdir(f'{BASE}/data'))       # phải thấy daily_series.csv và promotions.csv
print('CPU:', os.cpu_count())

## 1. Lấy code

Repo private thì nhập token GitHub khi được hỏi. Token chỉ nhập qua ô `getpass`, đừng dán vào cell. Clone xong, token được xóa khỏi cấu hình git.

In [ ]:
import getpass, subprocess
REPO = 'github.com/myuyen0304/retail-sales-forecasting-research.git'
if not os.path.exists('/content/rsf'):
    token = getpass.getpass('GitHub token (để trống nếu repo public): ')
    url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
    subprocess.run(['git', 'clone', '--depth', '1', '-q', url, '/content/rsf'], check=True)
    subprocess.run(['git', '-C', '/content/rsf', 'remote', 'set-url', 'origin', f'https://{REPO}'], check=True)
    del token, url
%cd /content/rsf
!git log -1 --oneline

## 2. Cài thư viện và kiểm code

Colab có sẵn numpy, pandas và torch, nhưng phiên bản khác laptop. Code mới chỉ được kiểm trên laptop (pandas 3.1). Nếu pytest báo lỗi, **dừng lại** và gửi toàn bộ thông báo lỗi về để sửa, đừng chạy tiếp.

In [ ]:
!pip -q install -e . statsmodels lightgbm xgboost prophet pyarrow
import importlib.metadata as md
print({p: md.version(p) for p in ['pandas', 'numpy', 'joblib', 'torch', 'scikit-learn']})
!python -m pytest -q

## 3. Chép tham số đã tune từ repo sang Drive

Tham số của 7 model thống kê và cây đã tune trên laptop và nằm trong `results/` của repo. Bước này chép chúng sang Drive. Không chép thì lệnh tune sẽ tìm không thấy và có thể chạy lại từ đầu.

In [ ]:
import json, shutil
R = os.environ['RSF_RESULTS_DIR']
os.makedirs(f'{R}/tuning', exist_ok=True)
for f in os.listdir('results/tuning'):
    if not os.path.exists(f'{R}/tuning/{f}'):
        shutil.copy(f'results/tuning/{f}', f'{R}/tuning/{f}')
repo_bp = json.load(open('results/best_params.json'))
drive_bp = json.load(open(f'{R}/best_params.json')) if os.path.exists(f'{R}/best_params.json') else {}
json.dump({**repo_bp, **drive_bp}, open(f'{R}/best_params.json', 'w'), indent=2)
print(sorted(json.load(open(f'{R}/best_params.json'))))

## 4. Tune RNN/LSTM (TOTAL / revenue, 4 fold, 9 cấu hình mỗi model)

In [ ]:
!python -m rsf.run tune --models rnn,lstm --jobs 2
!cat $RSF_RESULTS_DIR/best_params.json

## 5. Cross-validation (4 chuỗi × 2 target × 4 fold)

- RNN/LSTM bản chính: 5 seed, 160 task mỗi model.
- Bản `_ar`: 3 seed, 96 task mỗi model.

Bước này lâu nhất. Ô 6 bên dưới cho biết đã chạy được bao nhiêu task và ước tính thời gian còn lại.

In [ ]:
!python -m rsf.run cv --models rnn,lstm --jobs 2
!python -m rsf.run cv --models rnn_ar,lstm_ar --jobs 2 --seeds 3

## 6. Theo dõi tiến độ

Trong lúc cell ở mục 5 đang chạy thì không chạy được cell khác. Chạy cell này khi cell đó đã dừng hoặc bị ngắt. Nếu mở phiên mới, chạy lại các cell ở đầu notebook và mục 1–2 trước.

In [ ]:
import pandas as pd
from pathlib import Path
R = Path(os.environ['RSF_RESULTS_DIR'])
need = {'rnn': 160, 'lstm': 160, 'rnn_ar': 96, 'lstm_ar': 96}
done = {m: len([p for p in (R / 'preds/cv' / m).glob('*.parquet') if not p.stem.endswith('_fitted')]) for m in need}
print({m: f'{done[m]}/{need[m]}' for m in need})
if (R / 'runs_cv.csv').exists():
    log = pd.read_csv(R / 'runs_cv.csv')
    ok = log[log.status == 'ok']
    sec = ok.groupby('model').fit_s.median()
    print('Số giây fit (trung vị):', sec.round(0).to_dict())
    left = sum((need[m] - done[m]) * sec.get(m, sec.max() if len(sec) else 0) for m in need) / 2 / 3600
    print(f'Ước tính còn khoảng {left:.1f} giờ (với --jobs 2)')
    print('Task lỗi:', (log.status != 'ok').sum())

## 7. Mang kết quả về laptop

1. Trên Google Drive (web), chuột phải thư mục `MyDrive/rsf/results` → Download. Drive sẽ nén thành file zip.
2. Giải nén vào một thư mục riêng, ví dụ `D:\colab_rsf\results`.
3. Trên laptop, chạy trong PowerShell:
```
cd D:\retail-sales-forecasting-research
$env:PYTHONUTF8=1
.\.venv\Scripts\python.exe scripts\merge_colab.py D:\colab_rsf\results
.\.venv\Scripts\python.exe scripts\merge_colab.py D:\colab_rsf\results --apply
```
Lệnh đầu chỉ xem trước. Thấy số file đúng thì chạy lệnh thứ hai.